# RNNMorph PyTorch Training - Google Colab

This notebook trains the RNNMorph POS tagger using PyTorch on Google Colab with GPU acceleration.

## Features
- ✅ GPU acceleration with Tensor Cores
- ✅ Automatic Mixed Precision (AMP) for 2x speedup
- ✅ TensorBoard monitoring
- ✅ Google Drive checkpoint backup
- ✅ Progress bars with ETA

**Estimated training time:** ~3-4 hours for 50 epochs on full dataset (~7.8M words)

## 1. Setup Environment

In [ ]:
#@title Install Dependencies
!pip install torch tensorboard tqdm pymorphy2 russian-tagsets nltk h5py -q
print("✅ Dependencies installed")

In [ ]:
#@title Check GPU Availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: GPU not available! Go to Runtime > Change runtime type > GPU")

In [ ]:
#@title Mount Google Drive (Optional but Recommended)
from google.colab import drive
import os

drive.mount('/content/drive')

# Create checkpoint directory
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/rnnmorph_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")

## 2. Clone Repository and Download Data

In [ ]:
#@title Clone RNNMorph Repository
!git clone https://github.com/IlyaGusev/rnnmorph.git
%cd rnnmorph
print("✅ Repository cloned")

In [ ]:
#@title Download Training Data
!python download_training_data.py --all 2>&1 | tail -20
print("\n✅ Training data downloaded")

In [ ]:
#@title Check Available Training Files
import os

data_dir = 'rnnmorph/datasets/prepared'
files = os.listdir(data_dir)

print("Available training files:")
for f in sorted(files):
    path = os.path.join(data_dir, f)
    if os.path.isfile(path) and f.endswith('.txt'):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  📄 {f:<40} ({size:.1f} MB)")

## 3. Training Configuration

In [ ]:
#@title Training Hyperparameters

# Training data file
TRAIN_FILE = 'rnnmorph/datasets/prepared/training_combined.txt'  # Full dataset (~230MB, 7.8M words)
# TRAIN_FILE = 'rnnmorph/datasets/prepared/training_test_20k.txt'  # Small test (1MB, 20K words)

# Training parameters
EPOCHS = 50              # Number of training epochs
BATCH_SIZE = 64          # Batch size (reduce if OOM)
EXTERNAL_BATCH_SIZE = 5000  # External batch for bucketing
LEARNING_RATE = 0.001    # Initial learning rate
VAL_PART = 0.05          # Validation split (5%)

# Checkpointing
SAVE_FREQ = 1            # Save every N epochs
KEEP_LAST_N = 3          # Keep last N checkpoints

# Use Google Drive for checkpoints
USE_GOOGLE_DRIVE = True  # Set to False to save only in Colab

# Mixed Precision (2x speedup on GPU)
USE_AMP = True

# Output directory
OUTPUT_DIR = 'output'

print(f"Training configuration:")
print(f"  📊 Data file: {TRAIN_FILE}")
print(f"  🔄 Epochs: {EPOCHS}")
print(f"  📦 Batch size: {BATCH_SIZE}")
print(f"  🎯 Learning rate: {LEARNING_RATE}")
print(f"  💾 Checkpoints: {DRIVE_CHECKPOINT_DIR if USE_GOOGLE_DRIVE else OUTPUT_DIR}")
print(f"  ⚡ Mixed Precision: {'Enabled' if USE_AMP else 'Disabled'}")

## 4. Start Training

In [ ]:
#@title Run Training
import os

# Build command
cmd = f"""
python train_torch.py \\
    --train-file {TRAIN_FILE} \\
    --epochs {EPOCHS} \\
    --batch-size {BATCH_SIZE} \\
    --external-batch-size {EXTERNAL_BATCH_SIZE} \\
    --lr {LEARNING_RATE} \\
    --val-part {VAL_PART} \\
    --output-dir {OUTPUT_DIR} \\
    --save-freq {SAVE_FREQ} \\
    --keep-last-n {KEEP_LAST_N} \\
    --log-freq 10 \\
    --print-freq 20 \\
    --device cuda \\
    {'--use-amp' if USE_AMP else ''} \\
    {'--use-gdrive' if USE_GOOGLE_DRIVE else ''} \\
    --gdrive-path {DRIVE_CHECKPOINT_DIR if USE_GOOGLE_DRIVE else ''}
"""

print("Starting training...")
print(f"Command: {cmd}")
print("\n" + "="*70 + "\n")

!{cmd}

## 5. Monitor Training with TensorBoard

In [ ]:
#@title Launch TensorBoard
%load_ext tensorboard

log_dir = f'{OUTPUT_DIR}/logs'
print(f"TensorBoard logs: {log_dir}")
%tensorboard --logdir {log_dir}

## 6. Load and Test Trained Model

In [ ]:
#@title Load Best Checkpoint
import torch
import os

# Find best checkpoint
checkpoint_dir = f'{OUTPUT_DIR}/checkpoints'
best_checkpoint = os.path.join(checkpoint_dir, 'checkpoint_best.pt')

if os.path.exists(best_checkpoint):
    print(f"Loading best checkpoint: {best_checkpoint}")
else:
    # Find latest checkpoint
    checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')])
    if checkpoints:
        best_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])
        print(f"Loading latest checkpoint: {best_checkpoint}")
    else:
        print("No checkpoints found! Run training first.")
        best_checkpoint = None

if best_checkpoint:
    from rnnmorph.torch_train import load_model_from_checkpoint
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, train_config, build_config, gram_in, gram_out, word_vocab, char_set = \
        load_model_from_checkpoint(best_checkpoint, device=device)
    
    print(f"✅ Model loaded successfully!")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"   Device: {device}")

In [ ]:
#@title Test Inference
if best_checkpoint:
    from rnnmorph.batch_generator import BatchGenerator
    from pymorphy2 import MorphAnalyzer
    from russian_tagsets import converters
    import torch
    import numpy as np
    
    model.eval()
    morph = MorphAnalyzer()
    converter = converters.converter('opencorpora-int', 'ud14')
    
    # Test sentences
    test_sentences = [
        ['мама', 'мыла', 'раму'],
        ['кот', 'сидит', 'на', 'окне'],
        ['привет', ',', 'как', 'дела', '?'],
        ['я', 'люблю', 'программировать', 'на', 'питоне'],
    ]
    
    print("\n" + "="*70)
    print("Model Predictions")
    print("="*70)
    
    for sentence in test_sentences:
        print(f"\nSentence: {' '.join(sentence)}")
        print("-" * 50)
        
        # Prepare input
        word_indices, gram_vectors, char_vectors = BatchGenerator.get_sample(
            sentence=sentence,
            language='ru',
            converter=converter,
            morph=morph,
            grammeme_vectorizer=gram_in,
            max_word_len=30,
            word_vocabulary=word_vocab,
            word_count=10000,
            char_set=char_set
        )
        
        # Convert to tensors
        grammemes = torch.from_numpy(np.array(gram_vectors)).float().unsqueeze(0).to(device)
        chars = torch.from_numpy(np.array(char_vectors)).long().unsqueeze(0).to(device)
        
        # Forward pass
        with torch.no_grad():
            output = model(grammemes, chars)
            probs = torch.softmax(output[0, :, 1:], dim=-1)
            preds = probs.argmax(dim=-1)
        
        # Print results
        for i, word in enumerate(sentence):
            tag_idx = preds[i].item()
            tag_name = gram_out.get_name_by_index(tag_idx)
            score = probs[i, tag_idx].item()
            pos, tag = tag_name.split('#')
            print(f"  {word:15} → {pos:6} | {tag:30} (score: {score:.4f})")
    
    print("\n" + "="*70)

## 7. Download Checkpoints

In [ ]:
#@title Download Model Checkpoints
import os
from google.colab import files

checkpoint_dir = f'{OUTPUT_DIR}/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')]
    
    if checkpoints:
        print(f"Available checkpoints: {len(checkpoints)}")
        for cp in checkpoints:
            size = os.path.getsize(os.path.join(checkpoint_dir, cp)) / 1024 / 1024
            print(f"  📦 {cp:<40} ({size:.1f} MB)")
        
        # Download best checkpoint
        best_cp = os.path.join(checkpoint_dir, 'checkpoint_best.pt')
        if os.path.exists(best_cp):
            print(f"\nDownloading: checkpoint_best.pt")
            files.download(best_cp)
    else:
        print("No checkpoints found!")
else:
    print(f"Checkpoint directory not found: {checkpoint_dir}")

## 8. Training Tips

### Quick Test (5 minutes)
```python
TRAIN_FILE = 'rnnmorph/datasets/prepared/training_test_20k.txt'
EPOCHS = 5
BATCH_SIZE = 64
```

### Full Training (~4 hours on GPU)
```python
TRAIN_FILE = 'rnnmorph/datasets/prepared/training_combined.txt'
EPOCHS = 50
BATCH_SIZE = 64
USE_AMP = True  # Important for speed!
```

### If You Run Out of Memory
```python
BATCH_SIZE = 32  # Reduce batch size
GRAD_ACCUM = 2   # Use gradient accumulation
```

### Resume Interrupted Training
```python
# In a new cell:
!python train_torch.py --train-file {TRAIN_FILE} --resume {best_checkpoint}
```

---

## Troubleshooting

### GPU Not Available
1. Go to **Runtime** > **Change runtime type**
2. Select **GPU** under Hardware accelerator
3. Re-run all cells

### Out of Memory
- Reduce `BATCH_SIZE` to 32 or 16
- Enable `USE_AMP = True` for mixed precision

### Training Interrupted
- Checkpoints are saved to Google Drive
- Use `--resume checkpoint_best.pt` to continue

### Slow Training
- Ensure GPU is being used: check for `device: cuda` in output
- Enable mixed precision: `USE_AMP = True`
- Use full dataset only when ready (test with small file first)